In [1]:
library(data.table)

test_file <- "merged_sites_subset/CL-ACF_merged.csv"
dt_test <- fread(test_file)

str(dt_test)
names(dt_test)

Classes ‘data.table’ and 'data.frame':	52608 obs. of  58 variables:
 $ TIMESTAMP_START      :integer64 201801010000 201801010030 201801010100 201801010130 201801010200 201801010230 201801010300 201801010330 ... 
 $ TIMESTAMP_END        :integer64 201801010030 201801010100 201801010130 201801010200 201801010230 201801010300 201801010330 201801010400 ... 
 $ TA_F                 : num  8.18 8.02 7.85 7.11 6.37 ...
 $ TA_F_QC              : int  2 2 2 2 2 2 2 2 2 2 ...
 $ VPD_F                : num  1.864 1.753 1.641 1.285 0.929 ...
 $ VPD_F_QC             : int  2 2 2 2 2 2 2 2 2 2 ...
 $ P_F                  : num  0 0 0 0 0 0 0 0 0 0 ...
 $ P_F_QC               : int  2 2 2 2 2 2 2 2 2 2 ...
 $ TS_F_MDS_1           : num  -9999 -9999 -9999 -9999 -9999 ...
 $ TS_F_MDS_1_QC        : int  -9999 -9999 -9999 -9999 -9999 -9999 -9999 -9999 -9999 -9999 ...
 $ WS_F                 : num  5.15 4.78 4.4 4.18 3.96 ...
 $ WS_F_QC              : int  2 2 2 2 2 2 2 2 2 2 ...
 $ SWC_F_MDS_1          :

[1] "TIMESTAMP_START"       "TIMESTAMP_END"         "TA_F"                 
 [4] "TA_F_QC"               "VPD_F"                 "VPD_F_QC"             
 [7] "P_F"                   "P_F_QC"                "TS_F_MDS_1"           
[10] "TS_F_MDS_1_QC"         "WS_F"                  "WS_F_QC"              
[13] "SWC_F_MDS_1"           "SWC_F_MDS_1_QC"        "H_CORR"               
[16] "H_RANDUNC"             "H_RANDUNC_N"           "NEE_VUT_REF_RANDUNC_N"
[19] "LE_F_MDS"              "LE_F_MDS_QC"           "LE_CORR"              
[22] "LE_RANDUNC"            "LE_RANDUNC_N"          "H_F_MDS"              
[25] "H_F_MDS_QC"            "CO2_F_MDS"             "CO2_F_MDS_QC"         
[28] "NEE_VUT_REF"           "NEE_VUT_REF_QC"        "NEE_VUT_50"           
[31] "NEE_VUT_50_QC"         "NEE_VUT_REF_RANDUNC"   "GPP_NT_VUT_REF"       
[34] "GPP_DT_VUT_REF"        "GPP_NT_VUT_50"         "GPP_DT_VUT_50"        
[37] "RECO_NT_VUT_REF"       "RECO_DT_VUT_REF"       "RECO_NT_VUT_50"       
[40] "RECO_DT_VUT_50"        "SW_IN_F"               "SW_IN_F_QC"           
[43] "PPFD_IN"               "TA_F_MDS"              "TA_F_MDS_QC"          
[46] "VPD_F_MDS"             "SW_IN_F_MDS"           "SW_IN_F_MDS_QC"       
[49] "SW_IN_POT"             "LW_IN_F"               "NETRAD"               
[52] "USTAR"                 "WS"                    "G_F_MDS"              
[55] "G_F_MDS_QC"            "RH"                    "NIGHT"                
[58] "PA"

In [13]:
library(data.table)

# convert to character first
dt_test[, TIMESTAMP_START_chr := as.character(TIMESTAMP_START)]
dt_test[, TIMESTAMP_END_chr   := as.character(TIMESTAMP_END)]

# convert to POSIXct
dt_test[, TIMESTAMP_START_POSIX := as.POSIXct(TIMESTAMP_START_chr,
                                           format = "%Y%m%d%H%M",
                                           tz = "UTC")]

dt_test[, TIMESTAMP_END_POSIX := as.POSIXct(TIMESTAMP_END_chr,
                                         format = "%Y%m%d%H%M",
                                         tz = "UTC")]

# get range
range(dt_test$TIMESTAMP_START_POSIX)
range(dt_test$TIMESTAMP_END_POSIX)

[1] "2018-01-01 00:00:00 UTC" "2020-12-31 23:30:00 UTC"

[1] "2018-01-01 00:30:00 UTC" "2021-01-01 00:00:00 UTC"

In [1]:
library(data.table)
library(lubridate)

#1) read the sites 
current_sites <- fread("348EFP_Locations.csv")

# guess the site column
site_col <- if ("SITE_ID" %in% names(current_sites)) "SITE_ID" else names(current_sites)[1]
sites_keep <- unique(current_sites[[site_col]])

#2) list all half-hourly files and keep only those matching sites
flux_dir <- "merged_sites_subset_renamed"
all_files <- list.files(flux_dir, pattern = "_merged\\.csv$", full.names = TRUE)

# site id without _merged.csv
file_site_id <- gsub("_merged\\.csv$", "", basename(all_files))

files <- all_files[file_site_id %in% sites_keep]

cat(sprintf("Keeping %d / %d files (sites)\n", length(files), length(all_files)))

#3)
calc_site_year_climate_safe <- function(file, gs_tair_thresh = 5) {

  dt <- fread(file)

  site_id <- if ("SITE_ID" %in% names(dt)) unique(dt$SITE_ID)[1] else gsub("_merged.csv", "", basename(file))

  dt[, datetime := ymd_hm(as.character(TIMESTAMP_START))]
  dt[, year := year(datetime)]
  dt[, date := as.Date(datetime)]

  TA_use  <- if ("TA_F_MDS" %in% names(dt)) "TA_F_MDS" else if ("TA" %in% names(dt)) "TA" else NA_character_
  VPD_use <- if ("VPD_F_MDS" %in% names(dt)) "VPD_F_MDS" else if ("VPD" %in% names(dt)) "VPD" else NA_character_
  SW_use  <- if ("SW_IN_F_MDS" %in% names(dt)) "SW_IN_F_MDS" else if ("SW_IN" %in% names(dt)) "SW_IN" else NA_character_
  P_use   <- if ("P" %in% names(dt)) "P" else NA_character_

  gpp_candidates <- c("GPP_NT_VUT_50","GPP_NT","GPP_DT_VUT_50","GPP_DT")
  GPP_use <- gpp_candidates[gpp_candidates %in% names(dt)][1]
  has_gpp <- !is.na(GPP_use)

  daily <- dt[, .(
    GPP = if (has_gpp) mean(get(GPP_use), na.rm = TRUE) else NA_real_,
    VPD = if (!is.na(VPD_use)) mean(get(VPD_use), na.rm = TRUE) else NA_real_,
    TA  = if (!is.na(TA_use))  mean(get(TA_use),  na.rm = TRUE) else NA_real_,
    SW  = if (!is.na(SW_use))  mean(get(SW_use),  na.rm = TRUE) else NA_real_,
    P   = if (!is.na(P_use))   sum(get(P_use),    na.rm = TRUE) else NA_real_
  ), by = .(year, date)]

  # growing season definition:
  if (has_gpp) daily[, GS := GPP > 0] else daily[, GS := TA > gs_tair_thresh]

  yearly <- daily[, .(
    P_mm           = sum(P, na.rm = TRUE),
    Tair_mean_C    = mean(TA, na.rm = TRUE),
    SWin_mean_Wm2  = mean(SW, na.rm = TRUE),
    VPD_GS_hPa     = mean(VPD[GS], na.rm = TRUE) * 10  # assumes VPD is kPa
  ), by = year]

  yearly[, SITE_ID := site_id]
  setcolorder(yearly, c("SITE_ID","year","P_mm","Tair_mean_C","SWin_mean_Wm2","VPD_GS_hPa"))
  yearly[]
}

#4) run only for selected sites, with progress
out_list <- vector("list", length(files))
for (i in seq_along(files)) {
  cat(sprintf("[%d/%d] %s\n", i, length(files), basename(files[i])))
  out_list[[i]] <- calc_site_year_climate_safe(files[i])
}

climate_site_year <- rbindlist(out_list, fill = TRUE)

#5) save
dir.create("derived_tables", showWarnings = FALSE, recursive = TRUE)
fwrite(climate_site_year, "derived_tables/site_year_climate_metrics_selected_sites.csv")

cat("Saved: derived_tables/site_year_climate_metrics_selected_sites.csv\n")


Attaching package: ‘lubridate’


The following objects are masked from ‘package:data.table’:

    hour, isoweek, isoyear, mday, minute, month, quarter, second, wday,
    week, yday, year


The following objects are masked from ‘package:base’:

    date, intersect, setdiff, union




Keeping 348 / 521 files (sites)


In [2]:
library(data.table)
library(lubridate)

# 1) read the sites
current_sites <- fread("348EFP_Locations.csv")

# guess the site column
site_col <- if ("SITE_ID" %in% names(current_sites)) "SITE_ID" else names(current_sites)[1]
sites_keep <- unique(current_sites[[site_col]])

# 2) list all half-hourly files and keep only those matching sites
flux_dir <- "merged_sites_subset_renamed"
all_files <- list.files(flux_dir, pattern = "_merged\\.csv$", full.names = TRUE)

# site id without _merged.csv
file_site_id <- gsub("_merged\\.csv$", "", basename(all_files))
files <- all_files[file_site_id %in% sites_keep]

cat(sprintf("Keeping %d / %d files (sites)\n", length(files), length(all_files)))

# 3) function to calculate monthly meteorological summaries and return one row per site-year
calc_site_year_climate_monthly <- function(file, q_low = 0.10, q_high = 0.90) {
  
  dt <- fread(file)
  
  site_id <- if ("SITE_ID" %in% names(dt)) unique(dt$SITE_ID)[1] else gsub("_merged\\.csv$", "", basename(file))
  
  # parse timestamp
  dt[, datetime := ymd_hm(as.character(TIMESTAMP_START))]
  
  # fallback in case timestamp has seconds
  if (all(is.na(dt$datetime))) {
    dt[, datetime := ymd_hms(as.character(TIMESTAMP_START))]
  }
  
  dt <- dt[!is.na(datetime)]
  
  if (nrow(dt) == 0) {
    warning(sprintf("No valid timestamps in %s", file))
    return(NULL)
  }
  
  dt[, year := year(datetime)]
  dt[, month := month(datetime)]
  
  # choose meteorological columns
  TA_use  <- if ("TA_F_MDS" %in% names(dt)) "TA_F_MDS" else if ("TA" %in% names(dt)) "TA" else NA_character_
  VPD_use <- if ("VPD_F_MDS" %in% names(dt)) "VPD_F_MDS" else if ("VPD" %in% names(dt)) "VPD" else NA_character_
  SW_use  <- if ("SW_IN_F_MDS" %in% names(dt)) "SW_IN_F_MDS" else if ("SW_IN" %in% names(dt)) "SW_IN" else NA_character_
  P_use   <- if ("P" %in% names(dt)) "P" else NA_character_
  
  vars_available <- c(TA = TA_use, VPD = VPD_use, SW = SW_use, P = P_use)
  vars_available <- vars_available[!is.na(vars_available)]
  
  if (length(vars_available) == 0) {
    warning(sprintf("No climate variables found in %s", file))
    return(NULL)
  }
  
  out_list <- list()
  
  for (var_short in names(vars_available)) {
    
    var_col <- vars_available[[var_short]]
    
    tmp <- dt[, .(value = get(var_col)), by = .(year, month)]
    
    monthly <- tmp[, .(
      mean_val = if (all(is.na(value))) NA_real_ else mean(value, na.rm = TRUE),
      q10_val  = if (all(is.na(value))) NA_real_ else as.numeric(quantile(value, probs = q_low,  na.rm = TRUE, type = 7)),
      q90_val  = if (all(is.na(value))) NA_real_ else as.numeric(quantile(value, probs = q_high, na.rm = TRUE, type = 7))
    ), by = .(year, month)]
    
    monthly[, feature_mean := paste0(var_short, "_mean_m", sprintf("%02d", month))]
    monthly[, feature_q10  := paste0(var_short, "_q10_m", sprintf("%02d", month))]
    monthly[, feature_q90  := paste0(var_short, "_q90_m", sprintf("%02d", month))]
    
    mean_long <- monthly[, .(year, feature = feature_mean, value = mean_val)]
    q10_long  <- monthly[, .(year, feature = feature_q10,  value = q10_val)]
    q90_long  <- monthly[, .(year, feature = feature_q90,  value = q90_val)]
    
    out_list[[var_short]] <- rbindlist(list(mean_long, q10_long, q90_long))
  }
  
  out_long <- rbindlist(out_list, fill = TRUE)
  
  out_wide <- dcast(out_long, year ~ feature, value.var = "value")
  out_wide[, SITE_ID := site_id]
  setcolorder(out_wide, c("SITE_ID", "year", setdiff(names(out_wide), c("SITE_ID", "year"))))
  
  return(out_wide[])
}

# 4) run only for selected sites, with progress
out_list <- vector("list", length(files))

for (i in seq_along(files)) {
  cat(sprintf("[%d/%d] %s\n", i, length(files), basename(files[i])))
  out_list[[i]] <- calc_site_year_climate_monthly(files[i])
}

climate_site_year_monthly <- rbindlist(out_list, fill = TRUE)

# 5) save
dir.create("derived_tables", showWarnings = FALSE, recursive = TRUE)
fwrite(climate_site_year_monthly, "derived_tables/site_year_climate_monthly_metrics_selected_sites.csv")

cat("Saved: derived_tables/site_year_climate_monthly_metrics_selected_sites.csv\n")

Keeping 348 / 521 files (sites)


[1/348] AR-SLu_merged.csv
[2/348] AR-Vir_merged.csv
[3/348] AT-Neu_merged.csv
[4/348] AU-Boy_merged.csv
[5/348] AU-Emr_merged.csv
[6/348] AU-Gin_merged.csv
[7/348] AU-GWW_merged.csv
[8/348] AU-Lon_merged.csv
[9/348] AU-Rgf_merged.csv
[10/348] AU-Rig_merged.csv
[11/348] AU-Rob_merged.csv
[12/348] AU-Wac_merged.csv
[13/348] AU-War_merged.csv
[14/348] AU-Whr_merged.csv
[15/348] AU-Wom_merged.csv
[16/348] AU-Ync_merged.csv
[17/348] BR-CST_merged.csv
[18/348] BR-Npw_merged.csv
[19/348] BR-Sa3_merged.csv
[20/348] CA-Ca1_merged.csv
[21/348] CA-Ca2_merged.csv
[22/348] CA-Cbo_merged.csv
[23/348] CA-CF1_merged.csv
[24/348] CA-DB2_merged.csv
[25/348] CA-EM1_merged.csv
[26/348] CA-Gro_merged.csv
[27/348] CA-LP1_merged.csv
[28/348] CA-MA2_merged.csv
[29/348] CA-MA3_merged.csv
[30/348] CA-Man_merged.csv
[31/348] CA-NS1_merged.csv
[32/348] CA-NS2_merged.csv
[33/348] CA-NS3_merged.csv
[34/348] CA-NS4_merged.csv
[35/348] CA-NS5_merged.csv
[36/348] CA-NS6_merged.csv
[37/348] CA-NS7_merged.csv
[38/348] C